#### Import modules

In [ ]:
# Simple import setup
import sys
import os
sys.path.append('..')  # Add parent directory to path

# Import all necessary modules
import pandas as pd
import math
import matplotlib.pyplot as plt

# Import from the new modular structure
from src_code import (
    pf_enviroment, initialize_powerfactory, activate_project, list_and_select_study_case,
    list_and_activate_operation_scenario, run_contingency_analysis, process_cargabilidad, optimize_generators_for_substations,
    create_static_generator, calculate_power_limits, delete_generator, update_generator_power, cleanup_existing_generator
    )

### PowerFactory environment definition ------------

#### Initialize PowerFactory

In [4]:
# Añadir la ruta del entorno de PowerFactory
dig_path = r'C:\Program Files\DIgSILENT\PowerFactory 2021 SP2\Python\3.9'
pf_enviroment(dig_path)

PowerFactory environment initialized with path: C:\Program Files\DIgSILENT\PowerFactory 2021 SP2\Python\3.9


#### Get PowerFactory application

In [5]:
# Initialize PowerFactory application
app = initialize_powerfactory()

PowerFactory application connected successfully!


#### Activate project

In [6]:
project_name = "39 Bus New England System"
project = activate_project(app, project_name)

Proyecto '39 Bus New England System' activado con exito.


#### Activate study case

In [7]:
# Activate the desired study case
study_case_name = "1. Power Flow"  # Replace with your specific case name
active_study_case = list_and_select_study_case(app, study_case_name)

List of study cases:
2.1 Simulation Fault Bus 16 Stable
2.2 Simulation Fault Bus 16 Unstable
2.3 Simulation Fault Bus 31 Stable
2.4 Simulation Fault Bus 31 Unstable
2.5 Simulation Fault Line 2-3 Stable
2.6 Simulation Fault Line 2-3 Unstable
3. Small Signal Analysis (Eigenvalues)
4. EMT Simulation Fault Bus 03
1. Power Flow
Study case '1. Power Flow' activated.


#### Activate operation scenario

In [8]:
operation_scenario_name = "Basic Load Flow"
active_scenario = list_and_activate_operation_scenario(app, operation_scenario_name)

List of operation scenarios:
EMT
Basic Load Flow
Operation scenario 'Basic Load Flow' activated.


### Run contingency analysis ---------------

#### Get network data

In [9]:
project = app.GetActiveProject()
network_data = project.GetContents('Network Model.IntPrjfolder\\Network Data', 1)[0]
hoja = 'Grid'

#### Get buses

In [10]:
buses = app.GetCalcRelevantObjects('*.ElmTerm')
buses = [bus.loc_name for bus in buses]

#### Run search for max contingency

In [11]:
initial_potencia = 1
factor_potencia = 0.95
max_cargabilidad = 110
threshold_inconvergence = 10
df_results = optimize_generators_for_substations(
    app=app,  
    substations=buses,  # Should be a list of strings, not PowerFactory objects
    network_data=network_data, 
    hoja=hoja, 
    initial_potencia=initial_potencia, 
    factor_potencia=factor_potencia, 
    max_cargabilidad=max_cargabilidad, 
    threshold_inconvergence=threshold_inconvergence
)

Optimizando generador para la subestación 'Bus 01'.
Creando generador estático en la barra 'Bus 01' con potencia activa 1 MW y factor de potencia 0.95.
Buscando hoja 'Grid' en 'Network Data'.
Barra 'Bus 01' encontrada. Creando cubículo y generador estático.
Cubículo 'Cubicle_Gen_Bus 01' creado en la barra 'Bus 01' y conectado a la barra.
Error: No se pudo crear el generador estático 'Gen_Estatico_Bus 01' en la hoja 'Grid'.
Posibles causas:
1. El nombre ya existe
2. No hay permisos para crear objetos en esta ubicación
3. El tipo de objeto no es válido en este contexto
Error: No se pudo crear el generador en la subestación 'Bus 01'.
Optimizando generador para la subestación 'Bus 02'.
Creando generador estático en la barra 'Bus 02' con potencia activa 1 MW y factor de potencia 0.95.
Buscando hoja 'Grid' en 'Network Data'.
Barra 'Bus 02' encontrada. Creando cubículo y generador estático.
Cubículo 'Cubicle_Gen_Bus 02' creado en la barra 'Bus 02' y conectado a la barra.
Error: No se pudo cre

In [ ]:
#### Test the fixed generator creation


In [12]:
# Test creating a single generator to verify the fix works
test_bus = "Bus 01"
print(f"Testing generator creation for bus: {test_bus}")

# Get network data
project = app.GetActiveProject()
network_data = project.GetContents('Network Data', 1)[0]
hoja = 'Grid'

# Test creating a generator
voltage, p_gen, q_gen, static_generator, cubicle = create_static_generator(
    app, network_data, hoja, test_bus, 1, 0.95
)

if static_generator is not None:
    print(f"✅ SUCCESS: Generator created successfully!")
    print(f"   Bus voltage: {voltage}")
    print(f"   Active power: {p_gen} MW")
    print(f"   Reactive power: {q_gen} MVar")
    print(f"   Generator name: {static_generator.loc_name}")
    print(f"   Cubicle name: {cubicle.loc_name}")
    
    # Clean up the test generator
    print("Cleaning up test generator...")
    delete_generator(static_generator, cubicle)
    print("Test generator cleaned up.")
else:
    print("❌ FAILED: Generator creation failed!")


Testing generator creation for bus: Bus 01
Creando generador estático en la barra 'Bus 01' con potencia activa 1 MW y factor de potencia 0.95.
Buscando hoja 'Grid' en 'Network Data'.
Barra 'Bus 01' encontrada. Creando cubículo y generador estático.
Cubículo 'Cubicle_Gen_Bus 01' creado en la barra 'Bus 01' y conectado a la barra.
Ejecutando flujo de potencia para la barra 'Bus 01'.
Generador estático creado: Voltaje barra = 1.000000000001817, P generada = 1.0000000030814888, Q generada = -280.4560360580214
✅ SUCCESS: Generator created successfully!
   Bus voltage: 1.000000000001817
   Active power: 1.0000000030814888 MW
   Reactive power: -280.4560360580214 MVar
   Generator name: Gen_Estatico_Bus 01
   Cubicle name: Cubicle_Gen_Bus 01
Cleaning up test generator...


NameError: name 'delete_generator' is not defined